In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import pandas as pd

BASE_PATH = '/content/drive/MyDrive/Hyuga/'
month_order = {'April': 1, 'May': 2}

print("Loading CSVs...")
apr = pd.read_csv(BASE_PATH + 'april.csv', low_memory=False)
apr['month'] = 'May'        # file 'april.csv' actually contains May data
may = pd.read_csv(BASE_PATH + 'may.csv', low_memory=False)
may['month'] = 'April'      # file 'May.csv' actually contains April data

df = pd.concat([apr, may], ignore_index=True)
print(f"Total rows loaded: {len(df):,}")

# Step 1 — Find "Chat with Nutritionist" conversations
nutri_conv_ids = df[
    df['message_content'].astype(str).str.contains('Chat with Nutritionist', case=False, na=False)
]['conversation_id'].unique()
print(f"Unique conversations: {len(nutri_conv_ids):,}")

# Step 2 — Filter and preserve original index
conv_df2 = df[df['conversation_id'].isin(nutri_conv_ids)].copy()
conv_df2['created_at_content'] = pd.to_datetime(conv_df2['created_at_content'], errors='coerce')
conv_df2 = conv_df2.sort_values(['conversation_id', 'created_at_content']).reset_index()

# Step 3 — TRUE first incoming message per conversation
first_incoming = (
    conv_df2[
        (conv_df2['incoming_or_outcoming'] == 'Incoming') &
        (~conv_df2['message_content'].astype(str).str.contains('Chat with Nutritionist', case=False, na=False))
    ]
    .sort_values(['created_at_content', 'index'])
    .groupby('conversation_id')
    .first()
    .reset_index()[['conversation_id', 'message_content']]
    .rename(columns={'message_content': 'First Message'})
)

# Step 4 — One row per conversation that clicked Chat with Nutritionist
nutri_clicks = (
    conv_df2[
        conv_df2['message_content'].astype(str).str.contains('Chat with Nutritionist', case=False, na=False)
    ]
    .sort_values(['created_at_content', 'index'])
    .groupby('conversation_id')
    .first()
    .reset_index()
)

# Step 5 — Merge first message into clicks
result_df = nutri_clicks.merge(first_incoming, on='conversation_id', how='left')

# Step 6 — Tag Entry Points
def tag_entry_point(content):
    content = str(content).strip()
    if 'looking for Nutritionist advice' in content:
        return 'Entry Point 1'
    elif 'I need help with' in content:
        return 'Entry Point 2'
    else:
        return 'Other'

result_df['entry_point'] = result_df['First Message'].apply(tag_entry_point)

# Step 7 — Build ticket link
ACCOUNT_ID = 28052
result_df['Ticket Link'] = result_df['display_id'].apply(
    lambda x: f"https://app.limechat.ai/app/accounts/{ACCOUNT_ID}/conversations/{int(float(x))}"
    if pd.notna(x) else ''
)

# Step 8 — Filter ONLY EP1 and EP2
final = result_df[result_df['entry_point'] != 'Other'].rename(columns={
    'month': 'Month',
    'phone_number': 'Phone Number',
})[['Month', 'Phone Number', 'Ticket Link', 'First Message', 'entry_point']].copy()

final = final.rename(columns={'entry_point': 'Entry Type'})
final['Clicked Chat with Nutritionist'] = 'Yes'
final['month_order'] = final['Month'].map(month_order)
final = final.sort_values(['month_order', 'Entry Type']).drop(columns='month_order').reset_index(drop=True)

print(f"\nTotal rows (EP1 + EP2 only): {len(final):,}")
print(final.groupby(['Month', 'Entry Type'])['Phone Number'].count())

# Step 9 — Export
output_path = BASE_PATH + 'nutritionist_apr_may_ep1_ep2.xlsx'

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    final.to_excel(writer, sheet_name='EP1 EP2 Users', index=False)

print(f"\n✅ New file saved: {output_path}")
print(f"   Total rows: {len(final):,}")

Mounted at /content/drive
Loading CSVs...
Total rows loaded: 2,080,811
Unique conversations: 1,710

Total rows (EP1 + EP2 only): 1,055
Month  Entry Type   
April  Entry Point 1    130
       Entry Point 2    364
May    Entry Point 1    164
       Entry Point 2    396
Name: Phone Number, dtype: int64

✅ New file saved: /content/drive/MyDrive/Hyuga/nutritionist_apr_may_ep1_ep2.xlsx
   Total rows: 1,055


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import pandas as pd

BASE_PATH = '/content/drive/MyDrive/Hyuga/'
month_order = {'April': 1, 'May': 2}

print("Loading CSVs...")
apr = pd.read_csv(BASE_PATH + 'april.csv', low_memory=False)
apr['month'] = 'May'        # file 'april.csv' actually contains May data
may = pd.read_csv(BASE_PATH + 'may.csv', low_memory=False)
may['month'] = 'April'      # file 'May.csv' actually contains April data

df = pd.concat([apr, may], ignore_index=True)
df['created_at_content'] = pd.to_datetime(df['created_at_content'], errors='coerce')

# Step 0 — Date filter: Apr 1, 00:00 to May 31, 23:59:59
start_date = pd.Timestamp('2026-04-01 00:00:00')
end_date = pd.Timestamp('2026-05-31 23:59:59')
df = df[(df['created_at_content'] >= start_date) & (df['created_at_content'] <= end_date)].copy()
print(f"Total rows after date filter: {len(df):,}")

CONTENT_COL = 'message_content'

# Step 1 — Find "Chat with Nutritionist" conversations
nutri_conv_ids = df[
    df[CONTENT_COL].astype(str).str.contains('Chat with Nutritionist', case=False, na=False)
]['conversation_id'].unique()
print(f"Unique conversations (tickets): {len(nutri_conv_ids):,}")

# Step 2 — Filter and preserve original index
conv_df2 = df[df['conversation_id'].isin(nutri_conv_ids)].copy()
conv_df2 = conv_df2.sort_values(['conversation_id', 'created_at_content']).reset_index()

# Step 3 — TRUE first incoming message per conversation
first_incoming = (
    conv_df2[
        (conv_df2['incoming_or_outcoming'] == 'Incoming') &
        (~conv_df2[CONTENT_COL].astype(str).str.contains('Chat with Nutritionist', case=False, na=False))
    ]
    .sort_values(['created_at_content', 'index'])
    .groupby('conversation_id')
    .first()
    .reset_index()[['conversation_id', CONTENT_COL]]
    .rename(columns={CONTENT_COL: 'First Message'})
)

# Step 4 — One row per UNIQUE conversation (dedup multi-clicks)
nutri_clicks = (
    conv_df2[
        conv_df2[CONTENT_COL].astype(str).str.contains('Chat with Nutritionist', case=False, na=False)
    ]
    .sort_values(['created_at_content', 'index'])
    .groupby('conversation_id')
    .first()
    .reset_index()
)

# Step 5 — Merge
result_df = nutri_clicks.merge(first_incoming, on='conversation_id', how='left')

# Step 6 — Ticket link
ACCOUNT_ID = 28052
result_df['Ticket Link'] = result_df['display_id'].apply(
    lambda x: f"https://app.limechat.ai/app/accounts/{ACCOUNT_ID}/conversations/{int(float(x))}"
    if pd.notna(x) else ''
)

# Step 7 — Final shape (same format as before)
final = result_df.rename(columns={
    'month': 'Month',
    'phone_number': 'Phone Number',
})[['Month', 'Phone Number', 'Ticket Link', 'First Message']].copy()

final['Clicked Chat with Nutritionist'] = 'Yes'
final['month_order'] = final['Month'].map(month_order)
final = final.sort_values('month_order').drop(columns='month_order').reset_index(drop=True)

print(f"\nTotal unique tickets: {len(final):,}")
print(final.groupby('Month')['Phone Number'].count())

# Step 8 — Export
output_path = BASE_PATH + 'nutritionist_unique_tickets_apr_may.xlsx'

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    final.to_excel(writer, sheet_name='Unique Tickets', index=False)

print(f"\n✅ New file saved: {output_path}")

Mounted at /content/drive
Loading CSVs...
Total rows after date filter: 2,079,045
Unique conversations (tickets): 1,705

Total unique tickets: 1,705
Month
April    789
May      889
Name: Phone Number, dtype: int64

✅ New file saved: /content/drive/MyDrive/Hyuga/nutritionist_unique_tickets_apr_may.xlsx
